# MBG Tweet Classification — Case 1 BDC Internal 2026

**Tim:** Ein

**Anggota:** Muhamad Rifki Ardi Priadi / 1101223069

**Pendekatan:** TF-IDF (word + char n-gram) + Domain Keyword Features + Ensemble (Logistic Regression + LinearSVC)  
**Metric:** Balanced Accuracy  
**Dependencies:** `scikit-learn`, `pandas`, `numpy`, `openpyxl`


## 1. Import Libraries

In [30]:
import pandas as pd
import numpy as np
import re
import warnings
import os

from google.colab import drive
drive.mount('/content/drive')

warnings.filterwarnings("ignore")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.calibration import CalibratedClassifierCV
from scipy.sparse import hstack, csr_matrix

print("Libraries loaded...")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Libraries loaded


## 2. Konfigurasi Path & Label

In [31]:
# ── Konfigurasi path ──
LABELED_PATH  = '/content/drive/MyDrive/Dataset Case 1 BDC 2026_Tim ntah/case_1_labeled_data.xlsx'
PREDICT_PATH  = '/content/drive/MyDrive/Dataset Case 1 BDC 2026_Tim ntah/case_1_text_to_predict.xlsx'
TEMPLATE_PATH = '/content/drive/MyDrive/Dataset Case 1 BDC 2026_Tim ntah/case_1_template_sheet.xlsx'
OUTPUT_PATH   = "Ein.xlsx"

CLASSES = [
    "Anggaran", "Kualitas Pangan", "Distribusi", "Ekonomi",
    "Tata Kelola", "Sasaran Penerima", "Politik", "Lainnya"
]

print("Config set...")

Config set


## 3. Domain Keyword Lexicon

Membuat lexicon kata kunci per kelas berdasarkan domain knowledge program MBG.  
Fitur ini membantu model membedakan kelas yang semantically mirip (misal *Anggaran* vs *Tata Kelola*).


In [32]:
KEYWORDS = {
    "Anggaran": [
        "anggaran", "dana", "biaya", "rp ", "rupiah", "miliar", "triliun",
        "efisiensi", "defisit", "korupsi", "mark up", "markup", "uang negara",
        "apbn", "penggelapan", "subsidi", "pembiayaan", "keuangan", "duit",
        "bayar", "harga", "cost", "budget", "alokasi", "penggarongan",
        "bancakan", "nguras", "ngeruk", "pungutan", "ketidakefisienan",
    ],
    "Kualitas Pangan": [
        "kualitas", "gizi", "bergizi", "nutrisi", "makanan", "makan",
        "menu", "lauk", "nasi", "sayur", "buah", "protein", "kalori",
        "kedaluwarsa", "basi", "busuk", "bau", "tidak layak", "layak",
        "enak", "tidak enak", "hambar", "porsi", "sedikit",
        "keracunan", "mual", "sakit", "sehat", "tidak sehat",
        "minuman", "susu", "telur", "tempe", "tahu", "ikan", "daging",
        "sanitasi", "higienis", "bersih", "kotor", "cemaran",
    ],
    "Distribusi": [
        "distribusi", "salur", "penyaluran", "bagi", "bagikan",
        "daerah", "wilayah", "pelosok", "terpencil", "jangkau", "kirim",
        "logistik", "pengiriman", "armada", "kendaraan", "dapur umum",
        "satuan pelayanan", "sppg", "belum ada", "belum dapat",
        "tidak merata", "merata", "belum merata", "belum sampai",
        "belum diterima", "monitor", "pemantauan", "jangkauan",
    ],
    "Ekonomi": [
        "ekonomi", "lokal", "petani", "peternak", "nelayan", "umkm",
        "usaha", "pengusaha", "warung", "catering", "katering", "vendor",
        "rantai pasok", "suplai", "supplier", "lapangan kerja",
        "tenaga kerja", "pekerja", "pendapatan", "omzet", "keuntungan",
        "pertumbuhan", "industri", "produksi", "pertanian",
        "pangan lokal", "berdaya", "pemberdayaan", "penggerak ekonomi",
    ],
    "Tata Kelola": [
        "tata kelola", "manajemen", "regulasi", "aturan", "sop",
        "prosedur", "koordinasi", "instansi", "lembaga", "kementerian",
        "dinas", "bpom", "pengawasan", "evaluasi", "audit",
        "transparansi", "akuntabilitas", "laporan", "implementasi",
        "pelaksanaan", "kebijakan", "standar", "peraturan", "juknis",
    ],
    "Sasaran Penerima": [
        "siswa", "murid", "anak", "sekolah", "pelajar",
        "ibu hamil", "hamil", "balita", "baduta", "bayi", "lansia",
        "penerima", "sasaran", "manfaat", "penerima manfaat",
        "target", "peserta didik", "sd", "smp", "sma", "smk",
        "tk", "paud", "tidak dapat", "belum dapat", "sudah dapat",
        "dapat mbg", "terima mbg", "kelompok sasaran",
    ],
    "Politik": [
        "prabowo", "presiden", "jokowi", "anies", "politik",
        "pemerintah", "oposisi", "kampanye", "janji", "pemilu",
        "partai", "pdip", "gerindra", "golkar", "nasdem",
        "menteri", "dprd", "dpr", "legislatif",
        "kritik pemerintah", "gagal", "rezim", "penguasa",
        "wacana", "narasi", "tsm", "sistemik", "politis",
    ],
}

print(f"Keyword lexicon: {sum(len(v) for v in KEYWORDS.values())} total keywords")

Keyword lexicon: 209 total keywords


## 4. Text Preprocessing

Fungsi untuk membersihkan teks tweet:
- Lowercase
- Hapus URL, mention (`@user`), strip simbol hashtag
- Hapus karakter non-alfanumerik
- Normalisasi whitespace


In [33]:
def preprocess(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', ' url ', text)   # replace URLs
    text = re.sub(r'@\w+', ' user ', text)              # replace mentions
    text = re.sub(r'#(\w+)', r'\1', text)              # strip # symbol
    text = re.sub(r'[^a-z0-9\s]', ' ', text)           # remove special chars
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Quick test
sample = "@ARSIPAJA MBG di sekolah gw belum ada!! #MakanBergiziGratis https://t.co/abc123"
print("Original :", sample)
print("Processed:", preprocess(sample))

Original : @ARSIPAJA MBG di sekolah gw belum ada!! #MakanBergiziGratis https://t.co/abc123
Processed: user mbg di sekolah gw belum ada makanbergizigratis url


## 5. Keyword Feature Extractor

Untuk setiap tweet, hitung berapa keyword tiap kelas yang muncul,  
lalu normalisasi per baris &rarr; sparse matrix (n_samples × 8).

In [34]:
def keyword_features(texts: list, kw_dict: dict = KEYWORDS) -> csr_matrix:
    labels = list(kw_dict.keys())
    feat   = np.zeros((len(texts), len(labels)), dtype=np.float32)
    for i, text in enumerate(texts):
        t = str(text).lower()
        for j, lbl in enumerate(labels):
            feat[i, j] = sum(1 for kw in kw_dict[lbl] if kw in t)
    row_max = feat.max(axis=1, keepdims=True)
    row_max[row_max == 0] = 1
    return csr_matrix(feat / row_max)

print("keyword_features() defined...")

keyword_features() defined


## 6. Load & Explore Data

In [35]:
df_train    = pd.read_excel(LABELED_PATH)
df_predict  = pd.read_excel(PREDICT_PATH)
df_template = pd.read_excel(TEMPLATE_PATH)

print(f"Train  : {len(df_train)} baris")
print(f"Predict: {len(df_predict)} baris")
print()
print("Label distribution (train):")
print(df_train['label'].value_counts().to_frame('count').assign(
    pct=lambda d: (d['count'] / len(df_train) * 100).round(1)
))

Train  : 5000 baris
Predict: 1500 baris

Label distribution (train):
                  count   pct
label                        
Kualitas Pangan    1247  24.9
Politik             792  15.8
Anggaran            727  14.5
Lainnya             638  12.8
Tata Kelola         511  10.2
Sasaran Penerima    507  10.1
Distribusi          433   8.7
Ekonomi             145   2.9


In [36]:
# Preview data
print("Sample tweets per class:")
for lbl in CLASSES:
    sample = df_train[df_train['label'] == lbl]['full_text'].iloc[0]
    print(f"\n[{lbl}]")
    print(" ", sample[:120], "...")

Sample tweets per class:

[Anggaran]
  eiw efisiensi tai nyusahin huwekk kemana duitnya anjimg makan tuh mbg monyet ...

[Kualitas Pangan]
  @anakayahumi09 @irularr_ @ARSIPAJA Harus tetap dimakan karena namanya saja MBG, Makan Bersyukur Gratis ~~ ...

[Distribusi]
  🍱 Polsek Teluk Nibung monitoring Makan Bergizi Gratis SPPG Yayasan Al Muslimun Nusantara di Kapias Pulau Buaya.

👦👧 2.40 ...

[Ekonomi]
  @kiharani Kesejahteraan petani naik karena MBG andalkan pangan lokal dorongan Prabowo ...

[Tata Kelola]
  @vatuloh @DS_yantie Para koruptor dipastikan akan mendapatkan hukuman yg sesuai dgn perbuatannya. Lanjutkan MBG dgn peng ...

[Sasaran Penerima]
  @ARSIPAJA Pret. Di sekolah gw dapet MBG tetep aja anak-anaknya pada jajan lagi daaan ga semua suka sama menu MBGnya🥲 ...

[Politik]
  MBG bentuk penggarongan duit negara secara TSM (Terstruktur-Masif dan Sistempik). ...

[Lainnya]
  @tempodotco Buat MBG lah Anjayyy.... ...


## 7. Feature Engineering

Gabungkan 3 jenis fitur:
| # | Fitur | Alasan |
|---|-------|--------|
| 1 | TF-IDF Word n-gram (1–3) | Menangkap kata dan frasa penting |
| 2 | TF-IDF Char n-gram (3–5) | Robust terhadap typo & bahasa informal |
| 3 | Keyword features (8 dim) | Domain knowledge eksplisit per kelas |


In [37]:
# Preprocessing semua teks
df_train['clean']   = df_train['full_text'].apply(preprocess)
df_predict['clean'] = df_predict['full_text'].apply(preprocess)
df_train['lower']   = df_train['full_text'].str.lower().fillna('')
df_predict['lower'] = df_predict['full_text'].str.lower().fillna('')

# Word TF-IDF
word_vec = TfidfVectorizer(
    analyzer='word', ngram_range=(1, 3),
    min_df=2, max_df=0.95, max_features=100000, sublinear_tf=True,
)
Xtr_w = word_vec.fit_transform(df_train['clean'])
Xte_w = word_vec.transform(df_predict['clean'])
print(f"Word features  : {Xtr_w.shape[1]:,}")

# Char TF-IDF
char_vec = TfidfVectorizer(
    analyzer='char_wb', ngram_range=(3, 5),
    min_df=3, max_df=0.95, max_features=80000, sublinear_tf=True,
)
Xtr_c = char_vec.fit_transform(df_train['clean'])
Xte_c = char_vec.transform(df_predict['clean'])
print(f"Char features  : {Xtr_c.shape[1]:,}")

# Keyword features
Xtr_k = keyword_features(df_train['lower'].tolist())
Xte_k = keyword_features(df_predict['lower'].tolist())
print(f"Keyword features: {Xtr_k.shape[1]}")

# Gabungkan semua
X_train = hstack([Xtr_w, Xtr_c, Xtr_k])
X_test  = hstack([Xte_w, Xte_c, Xte_k])
y_train = df_train['label'].tolist()

print(f"\nTotal features : {X_train.shape[1]:,}")
print(f"Train shape    : {X_train.shape}")
print(f"Test shape     : {X_test.shape}")

Word features  : 18,455
Char features  : 24,481
Keyword features: 7

Total features : 42,943
Train shape    : (5000, 42943)
Test shape     : (1500, 42943)


## 8. Training Model + Cross-Validation

Menggunakan dua model dengan `class_weight='balanced'` agar secara eksplisit mengoptimasi **Balanced Accuracy**:
- **Logistic Regression** &rarr; probabilistik, stabil, mudah di-tune
- **LinearSVC** &rarr; kuat untuk fitur sparse/teks, dibungkus `CalibratedClassifierCV` agar menghasilkan probabilitas


In [38]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Model 1: Logistic Regression
lr = LogisticRegression(
    C=3.0, max_iter=3000, solver='lbfgs',
    class_weight='balanced', random_state=42, n_jobs=-1,
)
cv_lr = cross_val_score(lr, X_train, y_train, cv=cv,
                        scoring='balanced_accuracy', n_jobs=-1)
print(f"LR  — 5-CV Balanced Accuracy: {cv_lr.mean():.4f} ± {cv_lr.std():.4f}")

# Model 2: LinearSVC + Calibration
svc_base = LinearSVC(C=0.5, max_iter=5000, class_weight='balanced', random_state=42)
svc      = CalibratedClassifierCV(svc_base, cv=3)
cv_svc   = cross_val_score(svc, X_train, y_train, cv=cv,
                           scoring='balanced_accuracy', n_jobs=-1)
print(f"SVC — 5-CV Balanced Accuracy: {cv_svc.mean():.4f} ± {cv_svc.std():.4f}")

LR  — 5-CV Balanced Accuracy: 0.6094 ± 0.0216
SVC — 5-CV Balanced Accuracy: 0.5943 ± 0.0245


## 9. Fit Final Model (Full Training Data)

In [39]:
print("Training on full dataset...")
lr.fit(X_train, y_train)
svc.fit(X_train, y_train)
print("Selesai...")

Training on full dataset...
Selesai...


## 10. Ensemble Inference

Gabungkan probabilitas kedua model secara weighted average,  
dengan bobot = masing-masing CV score.


In [40]:
w_lr  = cv_lr.mean()
w_svc = cv_svc.mean()

proba_lr  = lr.predict_proba(X_test)
proba_svc = svc.predict_proba(X_test)

proba_ens   = (w_lr * proba_lr + w_svc * proba_svc) / (w_lr + w_svc)
pred_idx    = np.argmax(proba_ens, axis=1)
classes     = lr.classes_
predictions = [classes[i] for i in pred_idx]

print(f"Total prediksi: {len(predictions)}")
print(f"\nDistribusi prediksi:")
dist = pd.Series(predictions).value_counts()
for lbl, cnt in dist.items():
    asterisk = "*" * int(cnt / len(predictions) * 40)
    print(f"  {lbl:<20} {cnt:4d} ({cnt/len(predictions)*100:5.1f}%)  {asterisk}")

Total prediksi: 1500

Distribusi prediksi:
  Kualitas Pangan       367 ( 24.5%)  *********
  Politik               241 ( 16.1%)  ******
  Anggaran              227 ( 15.1%)  ******
  Lainnya               209 ( 13.9%)  *****
  Distribusi            140 (  9.3%)  ***
  Sasaran Penerima      130 (  8.7%)  ***
  Tata Kelola           124 (  8.3%)  ***
  Ekonomi                62 (  4.1%)  *


## 11. Simpan Output

In [41]:
df_out = df_template.copy()
df_out['label'] = predictions[:len(df_out)]

df_out.to_excel(OUTPUT_PATH, index=False)
print(f"File tersimpan : {OUTPUT_PATH}")
print(f"Shape          : {df_out.shape}")
print(f"\nPreview:")
display(df_out.head(10))

File tersimpan : Ein.xlsx
Shape          : (1500, 2)

Preview:


,id,label
0,TXT0001,Distribusi
1,TXT0002,Distribusi
2,TXT0003,Kualitas Pangan
3,TXT0004,Sasaran Penerima
4,TXT0005,Kualitas Pangan
5,TXT0006,Tata Kelola
6,TXT0007,Sasaran Penerima
7,TXT0008,Ekonomi
8,TXT0009,Kualitas Pangan
9,TXT0010,Tata Kelola


---
## Summary

| Model | CV Balanced Accuracy |
|-------|----------------------|
| Logistic Regression | `cv_lr.mean()` |
| LinearSVC | `cv_svc.mean()` |
| **Ensemble** | **max(cv_lr, cv_svc)** |
